# Backpropagation — Complete Notes
### (Part 1: The "How" — Code Implementation | Part 2: The "Why" — Mathematical Intuition)

---

## Preface

These notes are compiled from two lecture videos that are part of a "100 Days of Deep Learning" style playlist. Two videos before this one covered the **forward propagation** part of backpropagation (i.e., how a neural network makes a prediction). This document covers:

1. **The "How" video** — writing backpropagation from scratch in plain Python (no Keras), for both a **regression** problem and a **classification** problem, and then verifying the results by reproducing the exact same setup in **Keras**.
2. **The "Why" video** — the mathematical *intuition* behind why the backpropagation algorithm and the gradient descent update rule actually work — covering the concepts of derivatives, gradients, minima, and learning rate.

Together, these two videos are meant to give complete clarity — not just about *how* backpropagation is coded, but *why* it produces the correct weights and biases.

---

# PART 1 — THE "HOW": Implementing Backpropagation From Scratch

## 1.1 Goal of This Session

The plan is to take a neural network architecture (built and explained in earlier videos), and:

- Run backpropagation on **two datasets**:
  1. A **regression** dataset
  2. A **classification** dataset
- Implement everything using **plain Python/NumPy code** — **not** using Keras — so the underlying mechanics of backpropagation are fully transparent.
- After hand-coding it, **reproduce the same model in Keras** to confirm that both approaches converge to similar weights — acting as a sanity check/"referee" between the manual implementation and the library implementation.

---

## 1.2 The Regression Problem

### 1.2.1 Dataset

The dataset used previously (in the forward propagation video) is reused, with one change: the input column **CGPA** is replaced with **IQ**.

- Why this change matters: IQ typically ranges roughly from **70 to 130+** (a much larger numeric range), while **CGPA** ranges from **0 to 10**.
- **Rule of thumb for neural networks:** all input features should ideally be on a **similar scale/range**. When inputs are on wildly different scales, training becomes harder/slower. This is why feature scaling (standardization/normalization) matters in practice — the video uses the IQ example specifically to illustrate this point, even though scaling itself isn't applied here.

The dataset is stored as a **Pandas DataFrame**, with columns such as: student IQ, CGPA, and package (LPA) — used as the regression target.

### 1.2.2 Network Architecture

Same architecture as used in the earlier forward-propagation video:

- **Input layer:** 2 nodes (IQ, CGPA)
- **Hidden layer:** 2 nodes
- **Output layer:** 1 node (predicted package)

This is referred to throughout as the **"2 → 2 → 1"** architecture.

![Neural Network Architecture 2-2-1](images/diagram_1_architecture.png)

*Every weight (w111, w112, w121, w122, w2s, w221) and every bias (b11, b12, b21) shown above is one of the 9 trainable parameters referenced throughout these notes.*

### 1.2.3 Function 1 — `initialize_parameters`

**Purpose:** Given a neural network architecture (as a list, e.g. `[2, 2, 1]`), generate all the weight matrices and bias vectors needed, initialized to some starting values.

**Design choices used in the video:**
- All **weights** initialized to **0.1**
- All **biases** initialized to **0** (a valid alternative choice, but the instructor deliberately chose 0.1 for weights to avoid the "all-zero weights" symmetry problem, and used simple constants instead of random initialization purely for teaching clarity)

**How it works internally:**
- Given architecture `[2, 2, 1]`, the function creates 4 parameter objects:
  1. `W1` — weight matrix between input layer and hidden layer, shape determined by (previous layer size × current layer size). For `[2,2]` this becomes a matrix combining 4 individual weight values (the "4 numbers" the instructor points to on screen), e.g. `w111, w112, w121, w122`.
  2. `b1` — bias vector for the hidden layer (2 values, one per hidden neuron)
  3. `W2` — weight matrix between hidden layer and output layer
  4. `b2` — bias vector for the output layer (1 value)

- In short: `initialize_parameters(layer_dims)` → returns a dictionary/collection of all **W**'s and **b**'s for the given architecture, with every weight set to 0.1 and every bias set to 0.

> **Note:** The instructor mentions that instead of fixed constants, one could use NumPy's random initialization functions for a more realistic setup — but for teaching/debugging purposes, fixed values make it much easier to trace calculations by hand.

### 1.2.4 Function 2 — `linear_forward`

**Purpose:** Compute the output of **any single neuron**, given:
1. All the **weights** feeding into it
2. The **inputs** it receives (either raw input features, or outputs from the previous layer)
3. Its own **bias**

**Logic:** This is simply a **dot product** between the weight matrix and the input vector, plus the bias:

```
output = W.T · A_prev + b
```

This function is intentionally simple and is called repeatedly by the next function.

### 1.2.5 Function 3 — `forward_propagation` (Main Function)

This is the **main driver function** for the forward pass. Internally, it calls `linear_forward` layer by layer.

**Demonstration walkthrough (student #1):**
1. Extract the first row from the DataFrame → separate into `X` (input features: IQ, CGPA) and `y` (actual output: package).
2. Reshape `X` into the correct matrix shape (e.g., `(2,1)`).
3. Call `initialize_parameters` to get the current weights/biases.
4. Call `forward_propagation(X, parameters)`.

**Trace of what happens inside, for student 1:**
- The input layer receives `X` — this becomes the network's "A0" (activation of layer 0).
- These values get multiplied by `W1`, added to `b1` → produces **two** intermediate outputs, called `a11` and `a12` (the hidden layer's two activations, using a **linear activation function** in the regression case, meaning the raw weighted sum passes through unchanged).
- These hidden layer outputs (`a11`, `a12`) are then multiplied by `W2`, added to `b2` → produces the final single output `a2` (the predicted package).
- For student 1, the final prediction (`a2`) works out to approximately **0.32**.

**Return values:** The `forward_propagation` function returns **two things**:
1. `y_hat` (the final prediction, `a2`)
2. `A1` (the hidden layer's output — needed later)

**Why does it also return the hidden layer output?**
Because several **derivatives calculated during backpropagation depend on the output of the previous layer**. Storing `A1` at forward-pass time avoids recomputing it later.

### 1.2.6 Computing the Loss

Once `y_hat` is obtained:

```
Loss = (y − y_hat)²
```

This is the **Mean Squared Error (MSE)**-style loss used for the regression case (single squared-error term per data point, since we're processing one row/student at a time). For student 1, loss came out to roughly **13.something**.

### 1.2.7 Function 4 — `update_parameters`

**Purpose:** Update **all 9 parameters** of the network (4 weights `W1`, 4 for `W2`... — actually the total count depends on architecture, but in this network there are 9 trainable values: weights + biases across both layers) using the **gradient descent rule**:

```
param_new = param_old − (learning_rate × ∂Loss/∂param)
```

**Implementation details:**
- The derivatives (`∂Loss/∂param` for every weight and bias) were **already derived by hand in the previous video** — this function simply hardcodes those formulas.
- The function is described as looking "a bit scary" at first glance because it contains **9 separate update lines**, one per parameter — but conceptually it is just repeating the same update rule 9 times.
- **Learning rate used:** `0.001` — chosen because it gave the best results on this dataset.
- **Inputs required by this function:**
  - The current `y` (actual output, e.g. `package`, from the DataFrame, as a NumPy value)
  - `y_hat` (the prediction from forward propagation)
  - `A1` (the previous layer's output — both of its two node outputs)
  - The current `parameters` (weights and biases)
  - `X` (the original input)

**Effect after calling it:** All the parameters (weights and biases) get updated/modified. Running `update_parameters` once changes the values stored in `parameters` — confirmed by printing `parameters` before and after the call.

### 1.2.8 The Full Manual Training Loop (Regression, Step-by-Step Demo)

The instructor manually simulates **one epoch** (one full pass through all 4 students) to make the mechanics concrete:

```
Repeat for each student (row) in the dataset:
    1. Extract that student's X (inputs) and y (actual output)
    2. Call forward_propagation(X, parameters) → get y_hat, A1
    3. Compute loss = (y − y_hat)²
    4. Call update_parameters(y, y_hat, A1, parameters, X)
       → this updates all weights and biases immediately (i.e., updates happen 
         after every single row — this is essentially Stochastic Gradient Descent, 
         one point at a time, not batch)
```

- Student 1 → prediction ≈ 0.32 → parameters updated (small change).
- Student 2 → prediction changes slightly (parameters already shifted from step 1) → updated again.
- Student 3 → the instructor notes something interesting: student 3's CGPA/profile score is high, but package/placement outcome doesn't match expectations well → loss stays relatively "off" for this point.
- Student 4 → parameters updated again.

**Key structural point:** After the network sees **all 4 students once**, that constitutes **one epoch**. The **inner loop** runs once per student (4 times here); the **outer loop** runs once per epoch.

![Forward vs Backward Propagation](images/diagram_2_forward_backward.png)

*Forward propagation pushes data left→right to produce a prediction and loss; backpropagation pushes gradients right→left so `update_parameters` can correct every weight and bias.*

### 1.2.9 Average Loss Per Epoch

After completing the inner loop (all 4 students), the instructor computes:

```
average_loss = (L1 + L2 + L3 + L4) / 4
```

This average loss is what gets printed/tracked per epoch, so you can watch the loss decrease across epochs, e.g.:

- Epoch 1 → average loss ≈ **4.3ish**
- Epoch 2 → drops to around **~3**
- (continues decreasing across further epochs)

### 1.2.10 Full Vectorized/Looped Implementation (5 Epochs)

Instead of manually calling everything 5 times, the instructor writes a **compact single training loop**:

```python
parameters = initialize_parameters([2, 2, 1])
epochs = 5

for i in range(epochs):
    Loss = []

    for j in range(df.shape[0]):
        X = df[['iq', 'cgpa']].values[j].reshape(2, 1)
        y = df[['package']].values[j][0]

        # Forward propagation
        y_hat, A1 = forward_propagation(X, parameters)

        # Parameter update
        parameters = update_parameters(parameters, y, y_hat, A1, X)

        # Loss tracking
        Loss.append((y - y_hat)**2)

    print('Epoch -', i+1, 'Loss -', np.array(Loss).mean())
```

*(Reconstructed from the described logic — variable names approximate what was shown on screen.)*

**Observed output across 5 epochs:**
- Epoch 1 → average loss ≈ **25.3ish**
- Epoch 2 → parameters have adjusted enough that loss drops noticeably (close to the earlier point mentioned)
- Epoch 3 → loss further reduces to around **1.47**
- Epoch 5 → loss further drops to around **1.3-ish**

**Final result:** After training, the weights that started at `0.1` have moved to new, updated values — printed at the end to show the learned parameters.

---

## 1.3 Verifying the Manual Implementation Using Keras (Regression)

To make sure the from-scratch implementation is correct, the instructor rebuilds the **exact same** setup in **Keras**:

1. Same architecture: input(2) → hidden(2, linear activation) → output(1, linear activation)
2. Model summary confirms: **total 9 trainable parameters** (6 weights + 3 biases across the two layers) — matching the manual implementation's 9 parameters.
3. **Problem:** Keras randomly initializes weights by default, so to make a fair comparison:
   - Extract Keras's randomly initialized weights using `model.get_weights()`
   - Manually construct a **new weight list** where all weights = 0.1 and all biases = 0
   - Use `model.set_weights(new_weights)` to force Keras to start from the **same initial values** as the manual implementation.
4. Set the **optimizer's learning rate to 0.001** (same as the manual code).
5. Compile with **`loss='mean_squared_error'`**.
6. Call `model.fit(...)`.

**Result:** Keras converges to a loss around **1.47**, very close to the manually-coded result of **~1.34**. The small difference is attributed to Keras likely using a slightly different/more sophisticated **optimizer** internally (even though both are conceptually doing gradient descent).

**Conclusion of Part 1.2 (Regression):** The hand-written backpropagation algorithm was successfully converted into working code, verified against Keras, and both implementations behave consistently. This confirms the algorithm learned in the theory video is correctly implemented in practice.

---

## 1.4 The Classification Problem

Now the same exercise is repeated for a **classification** setting to solidify understanding.

### 1.4.1 Dataset

A new dataset is created with small modifications:
- **Inputs:** CGPA, Profile Score
- **Output:** Placement (binary — placed or not placed, i.e., 1/0)

### 1.4.2 Architecture

**Same architecture as before:** input(2) → hidden(2) → output(1)

### 1.4.3 Key Differences From the Regression Case

There are **two major changes** needed for classification:

**1. Activation Function:**
- In the regression example, **all** nodes (hidden and output) used a **linear** activation function.
- In the classification example, **every node** (all hidden nodes and the output node) now uses the **sigmoid** activation function.

**Sigmoid formula:**
```
σ(z) = 1 / (1 + e^(−z))
```

Sample walkthrough of forward propagation for student 1 with sigmoid:
```
z = (w1 × x1) + (w2 × x2) + b
a = sigmoid(z)
```
This `a` becomes the output/activation for that node, and this same pattern is applied at both the hidden layer and the output layer.

**2. Loss Function:**
- Regression used **Mean Squared Error (MSE)**.
- Classification uses **Binary Cross-Entropy** loss:

```
L = −[ y·log(ŷ) + (1−y)·log(1−ŷ) ]
```

### 1.4.4 Deriving the Gradients for Classification (By Hand)

Since the loss function and activation function changed, **all the derivatives must be recalculated** (the ones from the regression case no longer apply directly). The instructor re-derives these using the chain rule, exactly as done in the earlier theory video for regression — but now with sigmoid + binary cross-entropy.

**Setup — network naming convention:**
- `w2s`, `b21` etc. denote weights/bias into the final output node.
- `o11`, `o12` denote the hidden layer's two activations (post-sigmoid).
- `y_hat` = sigmoid of the final linear output = `a2` (also called `o21` in this section).

**Step 1: Derivative of Loss w.r.t. `y_hat`**

```
L = −y·log(ŷ) − (1−y)·log(1−ŷ)

∂L/∂ŷ = −(y/ŷ) + (1−y)/(1−ŷ)
       = (ŷ − y) / [ŷ(1−ŷ)]
```

(Derived step by step by differentiating each log term separately and combining over a common denominator.)

**Step 2: Derivative of `ŷ` (sigmoid output) w.r.t. `z` (the pre-activation value)**

Using the well-known identity for the sigmoid derivative:

```
∂ŷ/∂z = ŷ(1 − ŷ)     [since d/dz sigmoid(z) = sigmoid(z) × (1 − sigmoid(z))]
```

**Step 3: Multiply Step 1 and Step 2 (this is where the chain rule simplification happens)**

```
∂L/∂z = [ (ŷ − y) / (ŷ(1−ŷ)) ] × [ ŷ(1−ŷ) ]
       = (ŷ − y)
```

This is a **very clean and important result**: the combined derivative `∂L/∂z` for the output layer, under binary cross-entropy + sigmoid, simplifies to just **`(ŷ − y)`**. This mirrors the same simplification pattern seen in the regression case (which had `−2(y − ŷ)` at the equivalent stage) — the structure is analogous, just without the "×2" factor and with a sign flip.

**Step 4: Using this result to compute derivatives for `w2s` (weights into output), `w221`, and `b21`**

Because `z_final = w2s·o11 + w221·o12 + b21`:

```
∂z/∂w2s   = o11         (the corresponding previous-layer output)
∂z/∂w221  = o12
∂z/∂b21   = 1
```

So combining with Step 3:
```
∂L/∂w2s  = (ŷ − y) × o11
∂L/∂w221 = (ŷ − y) × o12
∂L/∂b21  = (ŷ − y)
```

**Step 5: Extending the chain rule backward into the hidden layer (for `w11`, `w12`, `b11`, `b12`, etc.)**

The same chain-rule pattern used in the regression derivation is reapplied here:
- The "already known" pieces (∂L/∂z at output layer) get reused.
- Multiply through by `∂z_output/∂o11` (or `o12`), which equals the corresponding weight (`w2s` or `w221`).
- Multiply by the local sigmoid derivative at the hidden node: `o(1−o)`.
- Multiply by the derivative of that hidden node's linear combination w.r.t. the specific weight (which equals the input feature, e.g., `x1` or `x2`).

Following this pattern for each of the remaining weights (`w11`, `w12`, `w121`, `w122`) and biases (`b11`, `b12`) — the structure stays consistent; **only the last multiplicative term changes** depending on which specific weight/bias is being differentiated (e.g., `x1` vs `x2`, or `1` for the bias term).

**Result:** All **9 derivatives** (for the classification case) are derived by hand, mirroring the same effort done previously for the regression case.

### 1.4.5 Coding the Classification Version

The code structure is **nearly identical** to the regression version, with exactly the **two changes** described above:
1. A `sigmoid` utility function is added:
   ```python
   def sigmoid(Z):
       return 1 / (1 + np.exp(-Z))
   ```
2. `linear_forward` now passes its output through `sigmoid(...)` before returning (instead of returning the raw linear combination).
3. `update_parameters` uses the newly derived classification-specific gradient formulas instead of the regression ones.

**Manual step-by-step run (classification):**
- Student 1 → loss ≈ 0.6-ish → parameters updated.
- Student 2 → loss slightly lower, parameters shift again.
- Student 3 → interesting case: this student has high CGPA and high profile score, but was **not placed** — so the model's prediction and the loss reflect this mismatch/error.
- Student 4 → parameters updated again.

**Full loop (5 epochs) is then run the same way as the regression case** (identical loop structure, just swapping in the classification-specific functions).

**A caveat/honest observation from the instructor:**
- Unlike the regression case, the classification loss **does not go below a certain point (~0.6)** even after training. Possible reasons discussed:
  - Very small dataset (only 4 points) — not enough data to learn a good decision boundary.
  - Possible issue with parameter initialization.
  - Possibly something off in the loss/derivative implementation.
  - The instructor is transparent that the exact cause isn't fully confirmed, but suspects it could be **any** of the above.

### 1.4.6 Verifying With Keras (Classification)

To check whether the "loss stuck at ~0.6" issue is a bug in the manual code or an inherent property of this tiny dataset:

1. Build the **same architecture** in Keras with sigmoid activations.
2. Set weights manually (same 0.1/0 initialization) to match the manual code.
3. Set learning rate and use **binary cross-entropy** loss.
4. Train the model.

**Result:** Keras **also** gets stuck around the same loss (~0.6) and does not improve further on this dataset. Since Keras's internal, well-tested implementation shows the **exact same behavior**, this confirms that:
- The manual implementation is **correct**.
- The plateau is a property of the **tiny, difficult dataset** (or its scale/init), **not** a bug in the from-scratch code.

### 1.4.7 Closing Remarks for Part 1

- The instructor strongly encourages viewers to **code along** with pen, paper, and their own machine while watching, since this material is fundamentally about **practice**, not passive viewing.
- This video (the "How") is meant to be paired with an upcoming video covering the **"Why"** part — described as the **most important video** in the whole backpropagation series.

---

# PART 2 — THE "WHY": Mathematical Intuition Behind Backpropagation

## 2.1 Purpose of This Video

This is explicitly called the **most conceptual** video in the 3-part backpropagation series (the other two covering "what" backpropagation does, and "how" to code it). If the previous two videos have already been watched, there should be no lingering confusion about backpropagation after this one.

**Shift in focus:** The earlier two videos focused on **how** the algorithm works mechanically. This video focuses on **why** it works — i.e., why the algorithm produces the *correct* result (correct weights and biases).

The backpropagation algorithm (recap):
1. Decide the number of **epochs**.
2. For each epoch, loop through every row in the dataset (this creates a **nested loop**: `epochs × number_of_rows`).
3. For each row: run **forward propagation** → get prediction (`ŷ`) → compute **loss** using the true value `y` (MSE for regression, binary cross-entropy for classification).
4. **Update all weights and biases** using the gradient descent rule.

The entire focus of this video: **why does subtracting the gradient, scaled by the learning rate, gradually give us the correct weights and biases?**

---

## 2.2 Core Concept #1 — The Loss Function is a Function of ALL Parameters

### 2.2.1 What Does "Function Of" Mean?

Basic reminder: if `y = f(x)`, it means that changing `x` causes a corresponding change in `y`. This is the fundamental meaning of one quantity being "a function of" another.

### 2.2.2 Is the Loss a Function of `y` or of the Weights?

Using the CGPA/Profile Score/Package dataset example, consider the loss formula:

```
L = (y − ŷ)²
```

- `y` is a value taken directly from the dataset (the target/actual value) — it is a **constant** for any given data point, not something the model controls.
- Therefore, **L is not fundamentally "a function of y"** in the sense that matters here (you could replace `y` with any constant number and the shape of the question doesn't change).
- The real, interesting question is: **what does `ŷ` (the prediction) depend on?**

### 2.2.3 Expanding `ŷ` — Showing It Depends on ALL the Network's Parameters

Assuming a linear activation everywhere for simplicity, `ŷ` for the given 2→2→1 architecture expands (via substitution) into a large expression involving:

```
ŷ = w2s · [ w111·x1 + w112·x2 + b11 ] + w221 · [ w121·x1 + w122·x2 + b12 ] + b21
```

*(Exact term names vary slightly across the transcript, but the structural point is what matters.)*

**Key observation:** When this full expression is expanded, `ŷ` turns out to be a function of:
- All 4 weights of the first layer (`w111, w112, w121, w122`)
- Both biases of the first layer (`b11, b12`)
- Both weights of the second/output layer (`w2s, w221`)
- The output bias (`b21`)

...**in addition to** the fixed inputs `x1` and `x2` (CGPA, Profile Score — constants for a given row).

### 2.2.4 Conclusion: Loss is a Function of ALL Trainable Parameters

Since:
- `ŷ` depends on all 9 parameters (weights + biases), and
- `L` depends on `ŷ`,

...it follows that:

```
L = f(w111, w112, w121, w122, b11, b12, w2s, w221, b21)
```

**The Loss function is a function of every single trainable parameter in the network — 9 things in this example, but in general, a function of ALL weights and biases in the network.**

**The core intuition to internalize:**
> If you change *any one* (or *all*) of these 9 values, the loss will change. Your goal is to search through this 9-dimensional space of parameters and find the specific combination of values that makes the loss **as small as possible** (minimum).

**Analogy used:** Think of the neural network as a "box" with knobs (the weights and biases). You can turn these knobs up or down. The entire training process is about turning these knobs in just the right way so that the loss becomes minimized.

---

## 2.3 Core Concept #2 — What Is a Gradient? (Derivative Basics)

### 2.3.1 Gradient = "Fancy Word for Derivative"

The instructor explicitly demystifies the term: **"Gradient is just a fancy word for derivative."** This is important because the entire algorithm is literally named **Gradient Descent**.

### 2.3.2 Single-Variable Derivative

If a function depends on **only one variable**, e.g.:

```
y = f(x) = x² + x
```

...then differentiating `y` with respect to `x` is called taking the **derivative**, denoted:

```
dy/dx
```

Example:
```
y = x² + 2x
dy/dx = 2x + 1
```

### 2.3.3 Multi-Variable Function → Called a "Gradient," Not a "Derivative"

When a function depends on **more than one variable**, e.g.:

```
z = f(x, y) = x² + y²
```

...then differentiating with respect to each variable **separately** (holding the others constant) gives **partial derivatives**:

```
∂z/∂x = 2x
∂z/∂y = 2y
```

**This collection of partial derivatives (with respect to every variable the function depends on) is what's called the Gradient**, often denoted using the **∇ (nabla/del)** symbol.

### 2.3.4 Applying This to the Network's Loss Function

Since the network's loss function `L` depends on **9 different parameters** (not just 1), it is a **highly complex, multi-variable mathematical function**. So when the algorithm needs to "differentiate" the loss, it must compute the **gradient** — i.e., 9 separate partial derivatives, one with respect to **each** weight and **each** bias.

**This is exactly what the phrase "calculating gradients" means in the context of neural networks.**

### 2.3.5 Geometric Intuition — Slope in Higher Dimensions

- If a function depends on **1 variable** (e.g. `y = f(x)`), its graph is a simple 2D curve, and the derivative at a point tells you the **slope of the tangent line** at that point.
- If a function depends on **2 variables** (e.g. `z = f(x, y) = x² + y²`), the graph becomes a **3D surface**. The gradient at a point on this surface tells you the slope of the surface **in each dimension separately** — how the surface tilts as you move along the `x` direction, and separately, how it tilts as you move along the `y` direction.
- For the network's loss function, which depends on **9 variables**, this generalizes to a "**9-dimensional slope**" — impossible to visualize directly, but conceptually identical: it tells you how the loss tilts/changes as you nudge each individual parameter.

**Bridge to calculus fundamentals:** The instructor notes that in school (from 11th grade onward), students learn to differentiate many different types of mathematical functions and memorize differentiation rules/formulas — but the **intuitive meaning** of "what does differentiation actually represent" is often skipped. To understand backpropagation, this intuition is essential — hence the next section.

---

## 2.4 Core Concept #3 — What a Derivative Actually Means (Rate of Change)

### 2.4.1 Definition: Derivative = Rate of Change

Notation: `dy/dx`, where `y` is a function of `x` (both are quantities/variables).

**Physical/intuitive meaning:** This is often referred to (especially in Physics) as the **"rate of change."** It captures: *if I change `x` slightly, how much does `y` change, and in which direction?*

### 2.4.2 Example

```
y = x²   →   dy/dx = 2x
```

If `x` increases from 1 unit to 2 units (i.e., `x` changed by 1 unit), then correspondingly `y` changes by `2× ` that amount → hence written as `dy/dx = 2x`, meaning: *the rate of change of `y` with respect to `x` at any point `x` is `2x`.*

### 2.4.3 Both Magnitude AND Sign Matter

The derivative's value tells you **two things simultaneously**:
1. **Magnitude** — how much the dependent quantity changes for a small change in the independent quantity.
2. **Sign** — the *direction* of that change:
   - If `dy/dx` is **positive**, increasing `x` by 1 unit causes `y` to increase.
   - If `dy/dx` is **negative**, increasing `x` by 1 unit causes `y` to decrease.

**Application to the network:** This same reasoning applies to a term like `∂L/∂w11` — this tells you: if you increase `w11` by a small amount, how (and in which direction) does the loss `L` change — both in **magnitude and sign**.

### 2.4.4 Computing a Derivative "At a Point"

Example: `y = x² + 2x`. Find `dy/dx` **at x = 5**.

**Steps:**
1. First differentiate generally: `dy/dx = 2x + 1`
2. Then substitute the specific point: `x = 5` → `dy/dx = 2(5) + 1 = 11`

**Interpretation:** "The rate of change of `y` with respect to `x`, *at the point x=5*, is 11" — meaning, near that specific point, a small increase in `x` causes `y` to increase roughly 11× as much.

---

## 2.5 Core Concept #4 — The Concept of Minima

### 2.5.1 Visual Intuition

Given a bowl-shaped (parabola-like) curve/graph, it's visually obvious where the **minimum point** (lowest point) is — humans can just *look* at the graph and point to it.

**The challenge:** An algorithm doesn't have "eyes" — it can't visually inspect a graph and point to the minimum. It needs a **mathematical, computable procedure** to find the minimum, without ever "seeing" the shape of the function.

### 2.5.2 The Classical Calculus Method (Single Variable)

The standard technique learned in school:
1. Take the derivative of the function.
2. Set the derivative **equal to zero**.
3. Solve for the variable — this gives the point where the function is at a minimum (or maximum — but in this context, minimum).

**Example:**
```
y = x²
dy/dx = 2x
Set: 2x = 0  →  x = 0
```
So the minimum of `y = x²` occurs at `x = 0`.

### 2.5.3 Extending to Multiple Variables

Example: `z = f(x, y) = x² + y²` (a 3D bowl-shaped surface).

To find the minimum:
1. Compute **both** partial derivatives: `∂z/∂x` and `∂z/∂y`
2. Set **both** equal to zero simultaneously.
3. Solve for `x` and `y`.

**Worked example:**
```
∂z/∂x = 2x = 0   →   x = 0
∂z/∂y = 2y = 0   →   y = 0
```
So the minimum of `z = x² + y²` occurs at `x = 0, y = 0`.

### 2.5.4 Applying This to the Neural Network's Loss Function

Since the loss function depends on **9 parameters**, minimizing it requires:
1. Computing all **9 partial derivatives** (the gradient).
2. Setting all 9 of them equal to zero.
3. Solving for each parameter's value.

**This is exactly what the network is trying to do:** find the specific combination of all weights and biases where the loss is minimized — i.e., search for the minimum point in a **9-dimensional parameter space**.

> *(In practice, for complex networks, this system of equations is far too complex to solve directly/analytically by setting derivatives to zero — which is exactly why the **iterative, numerical** gradient descent approach, discussed next, is used instead.)*

---

## 2.6 Putting It Together — Why the Gradient Descent Update Rule Works

### 2.6.1 Setting Up a Simplified Example

To build intuition, the instructor **freezes** all parameters except one — treating all weights and one bias as constant, and focusing only on a single bias, `b21`, to understand how one single parameter update works. (This simplification is temporary — once understood for one parameter, the same logic extends to all 9.)

Now, the loss `L` is treated as (for this simplified illustration) **a function of just `b21`**.

### 2.6.2 The Update Rule (Recap)

```
b21_new = b21_old − (learning_rate × ∂L/∂b21)
```

**The two things to understand:**
1. Why are we **subtracting**?
2. Why does the sign of the derivative matter so much here?

### 2.6.3 Understanding via the Graph of Loss vs. `b21`

Picture a 2D plot: **x-axis = `b21`**, **y-axis = Loss**. This graph has some bowl-like shape (not necessarily symmetric), with a minimum point somewhere.

**Goal:** Find the value of `b21` where the Loss is at its minimum.

**Case 1 — If `∂L/∂b21` is POSITIVE:**
- This means: as `b21` increases, the Loss also increases (they move in the same direction).
- Since the actual goal is to **reduce** the loss, this tells us we should move in the **opposite direction** — i.e., we should **decrease** `b21`.
- Since the derivative is positive, subtracting it (`b21_old − positive_value`) naturally **decreases** `b21`. ✅ Correct direction.

**Case 2 — If `∂L/∂b21` is NEGATIVE:**
- This means: as `b21` increases, the Loss **decreases** (inverse relationship).
- So to reduce the loss further, we should **increase** `b21`.
- Since the derivative is negative, subtracting a negative value (`b21_old − (negative_value)`) becomes **addition**, which naturally **increases** `b21`. ✅ Correct direction again.

**The elegant insight:** The **minus sign** in the update formula is doing something very smart — it automatically handles **both cases** correctly, without needing separate "if positive, do X; if negative, do Y" logic. **The entire direction-finding mechanism is essentially: "move in the negative direction of the gradient."**

![Gradient Descent Direction Intuition](images/diagram_3_gradient_descent.png)

*Whichever side of the minimum you start on, moving in the negative-gradient direction always points you toward the minimum — this is the entire justification for the "−" sign in the update rule.*

### 2.6.4 Graphical Walkthrough

Imagine the loss curve with respect to `b21`:
- Starting value: `b21 = 5` (for example), placing you at some point on the curve, away from the minimum.
- At that starting point, suppose the **slope (derivative) is positive** → this tells the algorithm to move in the negative direction (i.e., decrease `b21`) → moving toward the minimum. ✅
- If instead you had started at a point where `b21 = −5` (on the *other side* of the minimum), the slope there would be **negative** → the algorithm would then move in the positive direction (increase `b21`) — again correctly moving *toward* the minimum from the other side.

**Key takeaway:** Regardless of which side of the minimum you start on, computing the **negative of the gradient** and moving in that direction always points you **toward the minimum**. This is literally why the algorithm is called **Gradient Descent** — you are *descending* along the negative gradient direction.

### 2.6.5 How Big a Step Do You Take?

Question: once you know the *direction* to move, **how far** should you move in that direction?

**Answer (from the raw formula, before adding learning rate):** You move by an amount **exactly equal to the magnitude of the current slope** — i.e., `w_new = w_old − slope`. The steepness of the slope itself determines the step size.

This raises the next topic: **the learning rate**.

---

## 2.7 Core Concept #5 — The Role of the Learning Rate

### 2.7.1 The Problem Without a Learning Rate

If you take steps whose size is dictated purely by the (potentially large) slope value, you can **overshoot** dramatically:

**Illustrative example (`b21`):**
- Start: `b21 = −5`. Suppose at this point, slope (`∂L/∂b21`) = **−7**.
- Update: `b21_new = −5 − (−7) = −5 + 7 = 2` → jumps quite far, to `b21 = 2`.
- At `b21 = 2`, suppose the new slope evaluates to **+7**.
- Update: `b21_new = 2 − 7 = −5` → jumps back to where it started!

**Result:** The updates can **oscillate back and forth** indefinitely without ever settling near the minimum — or even worse, values can spiral and shoot far outside the reasonable range, diverging entirely instead of converging.

### 2.7.2 The Fix: Scale Down the Step With a Learning Rate

To prevent this oscillation, every update step is **scaled down** by multiplying the slope by a small factor — the **learning rate**, typically something like `0.1` or `0.01`.

**Revised update rule:**
```
param_new = param_old − (learning_rate × slope)
```

**Re-running the earlier example with a learning rate of 0.1:**
- Start: `b21 = −5`, slope = −7.
- Update: `b21_new = −5 − (0.1 × −7) = −5 + 0.7 = −4.3`
- Now instead of a huge jump, you take a much smaller, smoother step.
- As you get closer to the minimum, the slope naturally becomes smaller too, so subsequent steps shrink automatically, and you gradually converge to the minimum smoothly, without oscillating.

### 2.7.3 Trade-off: Too Large vs. Too Small Learning Rate

- **Learning rate too small:** Steps become extremely tiny, and the algorithm takes a **very long time** to converge to the minimum — training becomes very slow.
- **Learning rate too large:** Steps overshoot the minimum repeatedly, and the algorithm may **never converge** — it can oscillate wildly or even **diverge** (values growing larger and larger instead of settling down).

**Conclusion:** Choosing an appropriate learning rate is **extremely important** — it's emphasized multiple times as "very, very important" and "very, very crucial" to get right.

![Learning Rate Comparison](images/diagram_4_learning_rate.png)

*Too small → painfully slow convergence. Good → smooth, shrinking steps toward the minimum. Too large → the steps overshoot the minimum and grow larger each time, oscillating outward instead of converging.*

### 2.7.4 Live Demonstration Using a Visualization Tool

The instructor references a **Google-provided browser-based tool** (a well-known "playground"-style tool for visualizing gradient descent on a 2D loss surface) to demonstrate the learning-rate trade-off visually:

- With a **very small learning rate** (e.g., 0.001): steps are tiny, and it takes many iterations to reach the minimum — clearly slow to converge.
- With a **moderate learning rate** (e.g., 0.1): convergence happens noticeably faster, reaching close to the minimum in far fewer steps.
- With a **too-large learning rate** (e.g., 1.0+): the algorithm jumps to the opposite side of the curve, and the step sizes actually **grow** rather than shrink over iterations — a diverging pattern, clearly demonstrating instability.

This tool is used purely as a visual aid to reinforce the theoretical discussion above.

---

## 2.8 Core Concept #6 — How Many Times Do We Repeat the Update? (Convergence & Epochs)

### 2.8.1 The Real Stopping Criterion: Convergence

Recall the update formula:
```
w_new = w_old − (learning_rate × slope)
```

**Convergence**, formally, is the condition where the newly-computed parameter value becomes **very close to** the previous value — meaning the slope at that point has become close to **zero** (approaching the minimum, where by definition the slope is zero). In other words: once further updates stop meaningfully changing the parameters, you've converged, and continuing to update provides little to no further benefit.

### 2.8.2 The Practical Shortcut Used In Code

Technically, the *correct* stopping condition is to check for convergence directly (e.g., stop when the change between `w_old` and `w_new` drops below some small threshold). However, in practice (and in the code shown in the earlier "How" video), this is often simplified to:

> **"Run the loop for a fixed number of epochs (e.g., 100, or a few hundred), and trust that convergence will happen within that many iterations."**

This is acknowledged as a somewhat pragmatic/simplified approach ("a bit of a shortcut"), but it's very commonly used in practice since precisely detecting convergence can be fiddly, and a sufficiently large fixed epoch count usually gets close enough to the minimum.

---

## 2.9 Summary of the "Why" — Full Chain of Reasoning

1. The network's **Loss** is a mathematical function of **every trainable parameter** (all weights and biases) — this is the starting insight.
2. To **minimize** a multi-variable function, you need its **gradient** (the set of all partial derivatives) — this connects to why the algorithm computes derivatives with respect to each weight/bias.
3. A **derivative/gradient** tells you the **rate of change** — both magnitude and direction/sign — of the loss with respect to a parameter.
4. Moving in the **negative direction of the gradient** always moves you **toward** a minimum, regardless of which side of the minimum you're currently on — this is why the update rule **subtracts** the gradient.
5. Moving by the *raw* gradient value can cause **overshooting/oscillation**, so the step is scaled down using a **learning rate** — small enough to ensure smooth, stable convergence, but not so small that training becomes impractically slow.
6. This entire process (compute gradient → step in negative gradient direction, scaled by learning rate) is repeated over many **iterations/epochs** until the parameters **converge** to values where the loss is minimized (or, practically, until a fixed number of epochs has been run).

This is the complete mathematical and intuitive justification for why the **Gradient Descent / Backpropagation** algorithm is able to find weights and biases that minimize the loss function of a neural network.

---

## Closing Notes From the Instructor

- Strong encouragement to **practice by hand** — re-derive the gradients with pen and paper, and re-run the code personally — since genuine understanding of backpropagation comes from repetition and hands-on practice, not passive watching.
- A follow-up video is planned covering the **"What"** part of backpropagation (described as the most important video in this specific mini-series) — recommended as required viewing.
- The deep learning series continues beyond this point with further topics.